# Full Simulation (One Config, One Call)

Edit the config values below (dates, assets, weights, MC parameters), then run the execution cell.

In [1]:
from orchestration import *


config = SimulationConfig(
    market=MarketDataConfig(
        start="2018-04-03",
        end="2025-12-31",
        fred_series="SOFR",
        fred_is_percent=True,
    ),
    assets=[
        SyntheticLETFAssetConfig(
            id="SPY_3X_SYN",
            underlying_ticker="SPY",
            leverage=3.0,
            ter=0.0092,
            spread=0.0030,
        ),
        SpotAssetConfig(
            id="TLT_SPOT",
            ticker="TLT",
        ),
    ],
    portfolio=PortfolioConfig(
        target_weights={
            "SPY_3X_SYN": 0.60,
            "TLT_SPOT": 0.40,
        },
        initial_capital=100_000.0,
        rebalance_frequency_days=252,
        tolerance_band=0.05,
        capital_gains_tax_rate=0.26,
    ),
    monte_carlo=MonteCarloConfig(
        n_paths=5000,
        horizon_days=2520,
        method="bootstrap",
        distribution="normal",
        student_t_df=6.0,
        seed=42,
    ),
    metrics_ruin_threshold_fraction=0.10,
    use_mean_risk_free_for_metrics=True,
)

result = run_complete_simulation(config)

print("Wealth paths shape:", result.portfolio.wealth_paths.shape)
print("Simulated returns shape:", result.simulated_asset_returns.shape)
print("Metrics summary:")
display(result.metrics.summary)


/Users/giacomomaggiore/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Wealth paths shape: (5000, 2521)
Simulated returns shape: (5000, 2520, 2)
Metrics summary:


,mean,median,p5,p95
metric,,,,
CAGR,0.199603,0.187731,-0.007560,0.438857
MaxDrawdown,-0.564491,-0.553990,-0.760653,-0.404820
DrawdownDurationDays,943.598000,826.000000,371.000000,1967.000000
Sharpe,0.581533,0.581562,0.083474,1.069152
Sortino,0.732931,0.722880,0.099829,1.390696
UlcerIndex,28.226860,26.464329,16.482995,46.754694
Ruined,0.000000,0.000000,0.000000,0.000000
ProbabilityOfRuin,0.000000,0.000000,0.000000,0.000000


## Visual Diagnostics

Run this cell after the simulation cell to inspect path behavior and tail outcomes.

In [2]:
import numpy as np
import pandas as pd

from visuals import (
    plot_drawdown_chart,
    plot_spaghetti_paths,
    plot_terminal_wealth_distribution,
)

wealth_paths = result.portfolio.wealth_paths
terminal = wealth_paths[:, -1]

terminal_summary = pd.Series(
    {
        "min": float(np.min(terminal)),
        "p5": float(np.quantile(terminal, 0.05)),
        "median": float(np.median(terminal)),
        "mean": float(np.mean(terminal)),
        "p95": float(np.quantile(terminal, 0.95)),
        "max": float(np.max(terminal)),
    },
    name="TerminalWealth",
)

display(terminal_summary.to_frame())

fig_spaghetti = plot_spaghetti_paths(
    wealth_paths=wealth_paths,
    n_sample=100,
    seed=42,
    normalize_to_1=True,
    title="Monte Carlo Spaghetti Plot (Normalized)",
)

fig_terminal = plot_terminal_wealth_distribution(
    wealth_paths=wealth_paths,
    bins=60,
    title="Distribution of Terminal Wealth",
)

fig_drawdown = plot_drawdown_chart(
    wealth_paths=wealth_paths,
    drawdowns=result.metrics.drawdowns,
    title="Drawdown Chart: Median vs Worst Case",
)

fig_spaghetti.show()
fig_terminal.show()
fig_drawdown.show()


,TerminalWealth
min,1.996547e+04
p5,9.269209e+04
median,5.587040e+05
mean,1.159940e+06
p95,3.803429e+06
max,3.750477e+07
